# Regressions
This section addresses questions 3 to 5 concerning the modeling of demand for contracts associated with each lender.

# import required libraries

In [40]:
!pip install pyfixest
!pip install pyblp

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import random
from scipy.stats.mstats import winsorize
import os
import pandas as pd
import pyfixest as pf
import statsmodels.formula.api as smf
import numpy as np
import pyblp

# Define Paths

In [3]:
# All paths are relative to this notebook's folder (<project>/code)
input_path = "../data/processed"
output_path = "../output"

# Ensure that the tables directory exists and create it if not
output_tex_path = os.path.join(output_path, 'tables')
os.makedirs(output_tex_path, exist_ok=True)
# Ensure that the figures directory exists and create it if not
output_plot_path = os.path.join(output_path, 'figures')
os.makedirs(output_plot_path, exist_ok=True)

# Read filtered and cleaned data prepared in previous chapter

In [4]:
filtered_data = pd.read_csv(f"{input_path}/filtered_data.csv")
data = pd.read_csv(f"{input_path}/cleaned_data.csv")

<ipython-input-4-495a6dcde5df>:1: DtypeWarning: Columns (14,15,16) have mixed types. Specify dtype option on import or set low_memory=False.
  filtered_data = pd.read_csv(f"{input_path}/filtered_data.csv")


In [6]:
data.replace("NA", np.nan, inplace=True)
data.replace(to_replace=[None], value=np.nan, inplace=True)
data.replace("Exempt", "0", inplace=True)

data["income"] = data["income"].astype("float")
data["lender_credits"] = data["income"].astype("float")
data["loan_term"] = data["loan_term"].astype("float")
data["interest_rate"] = data["interest_rate"].astype("float")
data["applicant_sex"] = data["applicant_sex"].astype("category")

for col in ["rate_spread", "combined_loan_to_value_ratio", "fee_share"]:
    lower_bound = data[col].quantile(0.005)
    upper_bound = data[col].quantile(0.995)
    data[f"{col}_outlier"] = (
        (data[col] < lower_bound) | (data[col] > upper_bound)
    ).astype(int)

outlier_columns = [
    f"{col}_outlier"
    for col in ["rate_spread", "combined_loan_to_value_ratio", "fee_share"]
]
data = data[(data[outlier_columns].sum(axis=1) == 0)]

# Q2 Indicators

In [7]:
lar = (
    data.groupby(["lei", "market", "rejected"])
    .size()
    .div(data.groupby(["lei", "market"]).size())
    .reset_index(name="lar")
    .query("rejected == 0").drop(columns='rejected')
)

laa = (
    data.groupby(["lei", "market", "rejected"])["loan_amount"]
    .sum()
    .div(data.groupby(["lei", "market"])["loan_amount"].sum())
    .reset_index(name="laa")
    .query("rejected == 0").drop(columns='rejected')
)

las = (
    data
    .groupby(["market", "lei"])["loan_amount"]
    .sum()
    .div(data.groupby(["market"])["loan_amount"].sum())
    .reset_index(name="las")
)

alas = (
    data.query("rejected == 0")
    .groupby(["market", "lei"])["loan_amount"]
    .sum()
    .div(data.query("rejected == 0").groupby(["market"])["loan_amount"].sum())
    .reset_index(name="alas")
)

als = (
    data.query("rejected == 0")
    .groupby(["market", "lei"])
    .size()
    .div(data.query("rejected == 0").groupby(["market"]).size())
    .reset_index(name="als")
)

data["deb_to_income_value"] = data["debt_to_income_ratio"].map(
    {
        "36": 36,
        "41": 31,
        "30%-<36%": 33,
        "20%-<30%": 25,
        "44": 44,
        "48": 48,
        np.nan: np.nan,
        "<20%": 20,
        "43": 43,
        "37": 37,
        "42": 42,
        "49": 49,
        "39": 39,
        "38": 38,
        "47": 47,
        "40": 40,
        "46": 46,
        "45": 45,
        ">60%": 60,
        "50%-60%": 55,
        "0": 0,
    }
)

acr = (
    data.groupby(["lei", "market"])["deb_to_income_value"]
    .mean()
    .reset_index(name="acr")
)

merge_columns = ["lei", "market"]
data_meta = (
    data.merge(lar, on=merge_columns, how="left")
    .merge(laa, on=merge_columns, how="left")
    .merge(las, on=merge_columns, how="left")
    .merge(alas, on=merge_columns, how="left")
    .merge(als, on=merge_columns, how="left")
    .merge(acr, on=merge_columns, how="left")
)
data_bak = data.copy()
data = data_meta

# Q3.1

## Price models

In [8]:
price_model1 = pf.feols(
    """interest_rate ~ fee_share + combined_loan_to_value_ratio + loan_amount + lender_share + loan_term | market
    """,
    data=data,
)

price_model2 = pf.feols(
    """interest_rate ~ fee_share + combined_loan_to_value_ratio + loan_amount + lender_share + loan_term +
 | market + debt_to_income_ratio
    """,
    data=data,
)
price_model3 = pf.feols(
    """interest_rate ~ fee_share + combined_loan_to_value_ratio + loan_amount + lender_share + loan_term + | market + debt_to_income_ratio + C(applicant_sex)
    """,
    data=data,
)
price_model4 = pf.feols(
    """interest_rate ~ fee_share + combined_loan_to_value_ratio + loan_amount + lender_share + lar + loan_term + | market + debt_to_income_ratio + C(applicant_sex)
    """,
    data=data,
)
price_model5 = pf.feols(
    """interest_rate ~ fee_share + combined_loan_to_value_ratio + loan_amount + lender_share + lar + laa + alas + als + loan_term + | market + debt_to_income_ratio + C(applicant_sex)
    """,
    data=data,
)
price_model6 = pf.feols(
    """rate_spread ~ fee_share + combined_loan_to_value_ratio + loan_amount + lender_share + lar + laa + alas + als + loan_term + | market + debt_to_income_ratio + C(applicant_sex)
    """,
    data=data,
)
print(pf.etable([price_model1, price_model2, price_model3, price_model4, price_model5, price_model6], type="tex"))

\renewcommand\cellalign{t}
\begin{threeparttable}
\begin{tabular}{lccccccc}
\toprule
 & \multicolumn{5}{c}{interest_rate} & rate_spread \\
\cmidrule(lr){2-6} 
 & (1) & (2) & (3) & (4) & (5) & (6) \\
\midrule
\addlinespace
fee_share & \makecell{8.702** \\ (2.713)} & \makecell{8.670** \\ (2.862)} & \makecell{8.807** \\ (2.970)} & \makecell{8.748** \\ (3.046)} & \makecell{8.548** \\ (3.024)} & \makecell{12.547*** \\ (0.630)} \\
combined_loan_to_value_ratio & \makecell{0.005*** \\ (0.001)} & \makecell{0.005*** \\ (0.001)} & \makecell{0.005*** \\ (0.001)} & \makecell{0.005*** \\ (0.001)} & \makecell{0.005*** \\ (0.001)} & \makecell{0.010*** \\ (0.000)} \\
loan_amount & \makecell{0.000 \\ (0.000)} & \makecell{0.000 \\ (0.000)} & \makecell{0.000 \\ (0.000)} & \makecell{0.000 \\ (0.000)} & \makecell{0.000 \\ (0.000)} & \makecell{-0.000 \\ (0.000)} \\
lender_share & \makecell{-1.992 \\ (1.398)} & \makecell{-1.958 \\ (1.422)} & \makecell{-1.932 \\ (1.403)} & \makecell{-1.896 \\ (1.435)} & \makec

# Q3.2

## Rejection Models

In [9]:
formula = "rejected ~ combined_loan_to_value_ratio + loan_amount + lender_share + loan_term + C(applicant_sex) + C(market)"
model = smf.logit(formula, data=data)
results = model.fit()
print(results.summary().as_latex())

formula = "rejected ~ combined_loan_to_value_ratio + loan_amount + lender_share + lar + laa + alas + als + loan_term + C(applicant_sex) + C(market)"
model = smf.logit(formula, data=data)
results = model.fit()
print(results.summary().as_latex())

Optimization terminated successfully.
         Current function value: 0.283954
         Iterations 7
\begin{center}
\begin{tabular}{lclc}
\toprule
\textbf{Dep. Variable:}                         &     rejected     & \textbf{  No. Observations:  } &   1009991    \\
\textbf{Model:}                                 &      Logit       & \textbf{  Df Residuals:      } &   1009903    \\
\textbf{Method:}                                &       MLE        & \textbf{  Df Model:          } &        87    \\
\textbf{Date:}                                  & Tue, 12 Nov 2024 & \textbf{  Pseudo R-squ.:     } &   0.06184    \\
\textbf{Time:}                                  &     16:40:10     & \textbf{  Log-Likelihood:    } & -2.8679e+05  \\
\textbf{converged:}                             &       True       & \textbf{  LL-Null:           } & -3.0570e+05  \\
\textbf{Covariance Type:}                       &    nonrobust     & \textbf{  LLR p-value:       } &     0.000    \\
\bottomrule
\end{tabular}


# Q4 Share Models

In [10]:
shares_model1 = pf.feols(
    """
                        lender_share ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term | market
                        """,
    data=data,
)
shares_model2 = pf.feols(
    """
                        lender_share ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term | market + debt_to_income_ratio
                        """,
    data=data,
)
shares_model3 = pf.feols(
    """
                        lender_share ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term | market + C(applicant_sex) + debt_to_income_ratio
                        """,
    data=data,
)
shares_model4 = pf.feols(
    """
                        lender_share ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term + lar + laa | market + C(applicant_sex) + debt_to_income_ratio
                        """,
    data=data,
)
shares_model5 = pf.feols(
    """
                        lender_share ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term + lar | market + C(applicant_sex) + debt_to_income_ratio
                        """,
    data=data,
)
shares_model6 = pf.feols(
    """
                        las ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term + laa | market + C(applicant_sex) + debt_to_income_ratio
                        """,
    data=data,
)
shares_model7 = pf.feols(
    """
                        als ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term | market + C(applicant_sex) + debt_to_income_ratio
                        """,
    data=data,
)
shares_model8 = pf.feols(
    """
                        alas ~ interest_rate + fee_share + combined_loan_to_value_ratio + loan_amount + loan_term | market + C(applicant_sex) + debt_to_income_ratio
                        """,
    data=data,
)
print(pf.etable([shares_model1, shares_model2, shares_model3, shares_model4, shares_model5, shares_model6, shares_model7, shares_model8], type="tex"))

\renewcommand\cellalign{t}
\begin{threeparttable}
\begin{tabular}{lccccccccc}
\toprule
 & \multicolumn{5}{c}{lender_share} & las & als & alas \\
\cmidrule(lr){2-6} 
 & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) \\
\midrule
\addlinespace
interest_rate & \makecell{-0.000** \\ (0.000)} & \makecell{-0.000** \\ (0.000)} & \makecell{-0.000** \\ (0.000)} & \makecell{-0.000*** \\ (0.000)} & \makecell{-0.000*** \\ (0.000)} & \makecell{-0.000** \\ (0.000)} & \makecell{-0.000** \\ (0.000)} & \makecell{-0.000** \\ (0.000)} \\
fee_share & \makecell{-0.238** \\ (0.077)} & \makecell{-0.238** \\ (0.076)} & \makecell{-0.240** \\ (0.077)} & \makecell{-0.236** \\ (0.074)} & \makecell{-0.237** \\ (0.076)} & \makecell{-0.278** \\ (0.089)} & \makecell{-0.237** \\ (0.074)} & \makecell{-0.277** \\ (0.089)} \\
combined_loan_to_value_ratio & \makecell{-0.000 \\ (0.000)} & \makecell{-0.000 \\ (0.000)} & \makecell{-0.000 \\ (0.000)} & \makecell{-0.000 \\ (0.000)} & \makecell{-0.000 \\ (0.000)} & \makecell{-0.0

# Q5.1 Discrete Choice Model

In [11]:
grouped = data.groupby(["lei", "market"]).size().reset_index(name="count")
top5 = (
    grouped.sort_values(["market", "count"], ascending=[True, False])
    .groupby("market")
    .head(5)
)
data_top_lenders = top5.merge(data, on=["lei", "market"], how="left")
data_top_lenders = data_top_lenders[
    data_top_lenders["lei"].isin(
        data_top_lenders.groupby("lei").size().sort_values()[-5:].index.values
    )
].copy()

data_top_lenders["lender"] = data_top_lenders["lei"].astype("category").cat.codes
formula = "lender ~ interest_rate + loan_amount + fee_share + loan_term + income"
model = smf.mnlogit(formula, data=data_top_lenders)
results = model.fit()
print(results.summary().as_latex())

Optimization terminated successfully.
         Current function value: 0.911532
         Iterations 12
\begin{center}
\begin{tabular}{lclc}
\toprule
\textbf{Dep. Variable:}   &      lender      & \textbf{  No. Observations:  } &    118691    \\
\textbf{Model:}           &     MNLogit      & \textbf{  Df Residuals:      } &    118667    \\
\textbf{Method:}          &       MLE        & \textbf{  Df Model:          } &        20    \\
\textbf{Date:}            & Tue, 12 Nov 2024 & \textbf{  Pseudo R-squ.:     } &   0.09358    \\
\textbf{Time:}            &     16:58:13     & \textbf{  Log-Likelihood:    } & -1.0819e+05  \\
\textbf{converged:}       &       True       & \textbf{  LL-Null:           } & -1.1936e+05  \\
\textbf{Covariance Type:} &    nonrobust     & \textbf{  LLR p-value:       } &     0.000    \\
\bottomrule
\end{tabular}
\begin{tabular}{ccccccc}
   \textbf{lender=1}    & \textbf{coef} & \textbf{std err} & \textbf{z} & \textbf{P$> |$z$|$} & \textbf{[0.025} & \textbf{0.975]

## Q5.2 BLP Model

In [12]:
data_model = data_top_lenders[~data_top_lenders["interest_rate"].isna()]
data_model = data_model[~data_model["fee_share"].isna()]
data_model = data_model[~data_model["income"].isna()]
data_model["prices"] = data_model["interest_rate"]
data_model["market_ids"] = data_model["market"]
data_model["shares"] = data_model["lender_share"]
data_agent = data_model
data_model = data_model.drop_duplicates(subset=["lei", "market"])

x1_formulation = pyblp.Formulation(
    "prices + loan_amount + loan_term"
)
x2_formulation = pyblp.Formulation(
    "1 + prices"
)
loan_formulation = (x1_formulation, x2_formulation)
individual_formulation = pyblp.Formulation(
    "income"
)
problem = pyblp.Problem(loan_formulation, data_model, individual_formulation, data_agent, pyblp.Integration('monte_carlo', size=10, specification_options={'seed': 0})) # integration=pyblp.Integration('product', size=5))
# results = problem.solve(sigma=np.ones((2, 2)), pi=np.ones((2,2)))
results = problem.solve(sigma=np.ones((2, 2)) * 0.05, pi=np.ones((2,2)) * 0.05)
results

Initializing the problem ...
Initialized the problem after 00:00:00.

Dimensions:
 T    N    I    K1    K2    D    MD 
---  ---  ---  ----  ----  ---  ----
79   168  790   4     2     2    3  

Formulations:
       Column Indices:          0     1          2           3    
-----------------------------  ---  ------  -----------  ---------
 X1: Linear Characteristics     1   prices  loan_amount  loan_term
X2: Nonlinear Characteristics   1   prices                        
       d: Demographics          1   income                        
Solving the problem ...

Nonlinear Coefficient Initial Values:
Sigma:        1           prices      |  Sigma Squared:        1           prices      |   Pi:          1           income    
------  -------------  -------------  |  --------------  -------------  -------------  |  ------  -------------  -------------
  1     +5.000000E-02                 |        1         +2.500000E-03  +2.500000E-03  |    1     +5.000000E-02  +5.000000E-02
prices  +5.00

The model may be under-identified. The total number of unfixed parameters is 11, which is more than the total number of moments, 3. Consider checking whether instruments were properly specified when initializing the problem, and whether parameters were properly configured when solving the problem.



At least one error was encountered. As long as the optimization routine does not get stuck at values of theta that give rise to errors, this is not necessarily a problem. If the errors persist or seem to be impacting the optimization results, consider setting an error punishment or following any of the other suggestions below:
The fixed point computation of delta failed to converge. This problem can sometimes be mitigated by increasing the maximum number of fixed point iterations, increasing the fixed point tolerance, choosing more reasonable initial parameter values, setting more conservative parameter or share bounds, or using different iteration or optimization configurations.

GMM   Computation  Optimization   Objective   Fixed Point  Contraction  Clipped    Objective      Objective      Projected                                                                                                           
Step     Time       Iterations   Evaluations  Iterations   Evaluations  Shares 

Problem Results Summary:
GMM     Objective      Projected    Reduced Hessian  Reduced Hessian  Clipped  Weighting Matrix  Covariance Matrix
Step      Value      Gradient Norm  Min Eigenvalue   Max Eigenvalue   Shares   Condition Number  Condition Number 
----  -------------  -------------  ---------------  ---------------  -------  ----------------  -----------------
 2    +2.069022E-20  +3.523530E-25   -1.045556E-15    +1.072664E-15      2      +3.276690E+13      +1.695799E+22  

Cumulative Statistics:
Computation  Optimizer  Optimization   Objective   Fixed Point  Contraction
   Time      Converged   Iterations   Evaluations  Iterations   Evaluations
-----------  ---------  ------------  -----------  -----------  -----------
 00:01:12       Yes          3            15          58295       176029   

Nonlinear Coefficient Estimates (Robust SEs in Parentheses):
Sigma:         1             prices       |  Sigma Squared:         1             prices       |   Pi:           1           